# 06. LightGBM Regressor

In [ ]:
import sys
from pathlib import Path
import optuna
import pandas as pd
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

sys.path.insert(0, str(Path.cwd().parent))
from src.models import get_lightgbm, save_model
from src.evaluation import regression_metrics
from src.utils import set_seed, save_json

set_seed(42)
FEAT_DIR = Path('../data_features')
MODEL_DIR = Path('../models'); MODEL_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR = Path('../results/metrics'); RES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train = pd.read_parquet(FEAT_DIR / 'train.parquet')
val = pd.read_parquet(FEAT_DIR / 'val.parquet')
test = pd.read_parquet(FEAT_DIR / 'test.parquet')

TARGET = 'delay_hours'
DROP = [TARGET, 'trip_id', 'load_id', 'dispatch_date']

feature_cols = [c for c in train.columns if c not in DROP]
cat_cols = [c for c in feature_cols if train[c].dtype == 'object']

for c in cat_cols:
    train[c] = train[c].astype('category')
    val[c] = val[c].astype('category')
    test[c] = test[c].astype('category')

X_train, y_train = train[feature_cols], train[TARGET]
X_val, y_val = val[feature_cols], val[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]
print('Categorical features:', cat_cols)

## 1. Tune

In [ ]:
def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 1000, step=100),
        num_leaves=trial.suggest_int('num_leaves', 16, 255),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        feature_fraction=trial.suggest_float('feature_fraction', 0.6, 1.0),
        bagging_fraction=trial.suggest_float('bagging_fraction', 0.6, 1.0),
        bagging_freq=trial.suggest_int('bagging_freq', 1, 10),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 5.0, log=True),
    )
    m = get_lightgbm(**params)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(30, verbose=False)],
          categorical_feature=cat_cols)
    return mean_absolute_error(y_val, m.predict(X_val))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)
print(study.best_params)

## 2. Final + save

In [ ]:
best_model = get_lightgbm(**study.best_params)
best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
               callbacks=[lgb.early_stopping(50, verbose=False)],
               categorical_feature=cat_cols)

final_metrics = {
    'train': regression_metrics(y_train, best_model.predict(X_train)),
    'val':   regression_metrics(y_val,   best_model.predict(X_val)),
    'test':  regression_metrics(y_test,  best_model.predict(X_test)),
    'best_params': study.best_params,
}
save_model(best_model, MODEL_DIR / 'lightgbm.pkl')
save_json(final_metrics, RES_DIR / 'lightgbm.json')
final_metrics